# FlashAttention — Fast and Memory-Efficient Exact Attention with IO-Awareness (2022)

_Papers · Transformers & LLMs_

**Compute the SAME attention as before, but with tiling plus an online (running) softmax so the giant N-by-N attention table is never written to GPU memory — exact result, far less memory, big speedup.**

---

This notebook is a practice scaffold for the **FlashAttention — Fast and Memory-Efficient Exact Attention with IO-Awareness (2022)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn.functional as F

torch.manual_seed(0)

def standard_attention(Q, K, V):
    "The textbook way: build the full N x N score table, softmax it, weight values."
    d = Q.shape[-1]
    S = (Q @ K.transpose(-2, -1)) / (d ** 0.5)   # (N, N)  <-- O(N^2) table in memory
    P = S.softmax(dim=-1)
    return P @ V

def flash_attention(Q, K, V, block, rescale=True):
    "FlashAttention forward: tile over K,V blocks; online softmax with running max/sum. Algorithm 1."
    N, d = Q.shape
    O = torch.zeros(N, V.shape[-1])              # running output accumulator  (O(N) state)
    m = torch.full((N, 1), float('-inf'))        # running max per query row
    l = torch.zeros(N, 1)                         # running normalizer (softmax denom)
    for j in range(0, N, block):                 # walk column-blocks of K,V
        Kj, Vj = K[j:j+block], V[j:j+block]
        S = (Q @ Kj.transpose(-2, -1)) / (d ** 0.5)        # small block patch (never the full N x N)
        m_tilde = S.max(dim=-1, keepdim=True).values        # block max
        m_new = torch.maximum(m, m_tilde)                   # new running max
        P = torch.exp(S - m_new)                            # block weights at the new max
        l_tilde = P.sum(dim=-1, keepdim=True)               # block sum
        factor = torch.exp(m - m_new) if rescale else torch.ones_like(m)  # rescale OLD by exp(m - m_new)
        l = factor * l + l_tilde                            # corrected running normalizer
        O = factor * O + P @ Vj                             # corrected running output
        m = m_new                                           # advance running max
    return O / l                                            # normalize ONCE at the end

# ---- (1) EXACTNESS: flash == standard, to floating-point tolerance (Theorem 1) ----
N, d = 64, 16
Q, K, V = torch.randn(N, d), torch.randn(N, d), torch.randn(N, d)
out_std = standard_attention(Q, K, V)
out_flash = flash_attention(Q, K, V, block=8)
print("max |flash - standard| =", (out_flash - out_std).abs().max().item())
print("torch.allclose(flash, standard):", torch.allclose(out_flash, out_std, atol=1e-6))  # True

# ---- (2) WORKED EXAMPLE recomputed: scores [1,3,2,0], values [10,20,30,40], blocks of 2 ----
Sx = torch.tensor([[1., 3., 2., 0.]])
Vx = torch.tensor([[10.], [20.], [30.], [40.]])
# treat Sx as precomputed scores: emulate flash over two blocks of 2
m, l, O = torch.tensor([[float('-inf')]]), torch.tensor([[0.]]), torch.tensor([[0.]])
for j in range(0, 4, 2):
    Sj = Sx[:, j:j+2]
    m_t = Sj.max(-1, keepdim=True).values
    m_n = torch.maximum(m, m_t)
    P = torch.exp(Sj - m_n)
    fac = torch.exp(m - m_n)
    l = fac * l + P.sum(-1, keepdim=True)
    O = fac * O + P @ Vx[j:j+2]
    m = m_n
print("worked-example flash output:", round((O / l).item(), 2),
      "| full-softmax:", round((Sx.softmax(-1) @ Vx).item(), 2))   # both 22.14

# ---- (3) MEMORY: standard materializes O(N^2); flash keeps O(N) running state ----
for n in (128, 256, 512, 1024):
    std_table = n * n            # entries in the N x N score table (standard)
    flash_state = 2 * n + n * d  # m,l (length n) + output (n x d)  -> O(N)
    print(f"N={n:>4}: standard table={std_table:>8} entries | flash state={flash_state:>6} (O(N))")

# ---- (4) ABLATION: remove the rescale -> NOT exact anymore ----
bad = flash_attention(Q, K, V, block=8, rescale=False)
print("no-rescale max |flash - standard| =", round((bad - out_std).abs().max().item(), 4),
      "| allclose:", torch.allclose(bad, out_std, atol=1e-6))   # large error, False

## Visualize the data & results

_How does attention memory grow with sequence length N? Standard attention stores the full N x N score table — O(N^2). FlashAttention keeps only a running max, running sum, and running output — O(N). And does the tiled online-softmax give the EXACT same answer as standard attention?_

In [ ]:
import torch

d = 64
print("N | standard N^2 | flash 2N+Nd (O(N))")
for N in (128, 256, 512, 1024, 2048):
    standard = N * N            # full score table
    flash = 2 * N + N * d       # running m,l + output -> O(N)
    print(f"{N:>4} | {standard:>10} | {flash:>8}")
# standard: 16384, 65536, 262144, 1048576, 4194304   (quadratic in N)
# flash:     8448, 16896,  33792,   67584,  135168    (linear in N)

# exactness check (Theorem 1): flash == standard
torch.manual_seed(0)
N, d = 64, 16
Q, K, V = torch.randn(N, d), torch.randn(N, d), torch.randn(N, d)
def std(Q,K,V):
    S = (Q @ K.T) / (Q.shape[-1] ** 0.5); return S.softmax(-1) @ V
def flash(Q,K,V,b=8):
    N,d = Q.shape; O=torch.zeros(N,d); m=torch.full((N,1),-1e30); l=torch.zeros(N,1)
    for j in range(0,N,b):
        S=(Q@K[j:j+b].T)/(d**0.5); mt=S.max(-1,keepdim=True).values; mn=torch.maximum(m,mt)
        P=torch.exp(S-mn); f=torch.exp(m-mn); l=f*l+P.sum(-1,keepdim=True); O=f*O+P@V[j:j+b]; m=mn
    return O/l
print("allclose:", torch.allclose(flash(Q,K,V), std(Q,K,V), atol=1e-6))  # True

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A query row's scores are $[2, 5]$ for block A and $[6, 1]$ for block B (process A then B). Track the running max $m$ and explain what the rescale factor does when block B arrives.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Block A: block max $\tilde m=5$, so running $m=5$. — _First block sets the running max (Section 3.1)._
- Block B: block max $\tilde m=6$, so $m^{\text{new}}=\max(5,6)=6$. — _New running max is the larger of old and block max (Algorithm 1, line 11)._
- Rescale factor for block A's old quantities $=e^{m-m^{\text{new}}}=e^{5-6}=e^{-1}=0.368$. — _Block A was summed against max 5; we correct it down to the new max 6._

**Answer:** Running max goes $5\to6$ when block B arrives. The old $\ell$ and $O$ from block A are multiplied by $e^{5-6}=0.368$, shrinking them to the new reference max before block B's contribution is added — keeping the result exactly equal to full-row softmax.

</details>

**Problem 2.** Standard attention stores the full $N\times N$ score table. For $N=1024$ vs $N=2048$ (float32, 4 bytes), how much memory does that table take, and how does FlashAttention's $O(N)$ running state compare?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $N=1024$: $1024^2\times4 = 4{,}194{,}304$ bytes $\approx 4$ MB. — _The score table is $N^2$ floats — $O(N^2)$ (standard attention)._
- $N=2048$: $2048^2\times4 = 16{,}777{,}216$ bytes $\approx 16$ MB. — _Doubling $N$ quadruples $N^2$ — the quadratic wall._
- FlashAttention keeps only $m,\ell$ (length $N$) plus output ($N\times d$): $O(N)$, so doubling $N$ only doubles it. — _Theorem 1: $O(N)$ additional memory._

**Answer:** The standard table jumps from ~4 MB to ~16 MB (a 4x jump for a 2x sequence). FlashAttention's running state grows only linearly, so it about doubles. That $O(N^2)$-vs-$O(N)$ gap is exactly what the CODEVIZ memory plot shows.

</details>

**Problem 3.** Ablation: you delete the line that rescales the running output ($O\leftarrow e^{m-m^{\text{new}}}O+\tilde P V_j$) and instead just do $O\leftarrow O+\tilde P V_j$. Will torch.allclose(flash, standard) still pass? Why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Without the rescale, blocks processed before a max-raising block keep their old (too-large) exponential scale. — _Their weights $e^{x-m_{\text{old}}}$ are never corrected to the new max $m^{\text{new}}$._
- The accumulated $O$ then mixes weights at inconsistent reference maxima, so it no longer equals full-row softmax. — _Online softmax is exact only because every term is expressed relative to the current max._
- torch.allclose returns False (unless no block ever raises the max, e.g. scores are non-increasing across blocks). — _The correctness in Theorem 1 depends on the rescale._

**Answer:** It fails: dropping the rescale leaves earlier blocks at the wrong exponential scale, so the running output no longer matches standard attention. The CODE ablation prints the (large) error — the rescale is precisely what makes FlashAttention exact.

</details>